# 02c Final Static Test Evaluation

Aim of this notebook: apply the frozen primary K-means NHS to the test-account partition without refitting or reopening model selection.

The test results are interpreted as a final held-out evaluation of the clean pipeline, with the caveat that the same split had been inspected during earlier exploratory work.


## 0. Configuration


In [1]:
from pathlib import Path
import hashlib
import json
import pickle

import duckdb
import numpy as np
import pandas as pd

from sklearn.metrics import average_precision_score, roc_auc_score

import nhs_split_eval as se


FEATURE_PATH = Path("static_features_v2/feat_final.parquet")
FEATURE_META_PATH = Path("static_features_v2/feature_build_meta.json")
ACCOUNT_SPLIT_PATH = Path("account_split_static.json")
REPRESENTATION_META_PATH = Path(
    "eda_outputs_01b_fit_features_core5/representation_meta.json"
)

MODEL_DIR = Path("static_model_comparison")
CAL_METRICS_PATH = MODEL_DIR / "calibration_model_metrics.csv"
SCORERS_PATH = MODEL_DIR / "frozen_generic_scorers.pkl"
MODEL_META_PATH = MODEL_DIR / "model_comparison_meta.json"

OUT = Path("static_test_evaluation")
OUT.mkdir(exist_ok=True)

TEST_METRICS_PATH = OUT / "test_kmeans_metrics.csv"
CAL_VS_TEST_PATH = OUT / "Table_04_calibration_vs_test_kmeans.csv"
TEST_STABILITY_PATH = OUT / "test_kmeans_stability.csv"
META_PATH = OUT / "final_test_meta.json"

PRIMARY_METHOD = "kmeans_core5"
STATIC_EVAL_WIN = 0
KS = (10, 30, 50, 100)

CORE5 = [
    "period_strength",
    "iat_cv",
    "log_volume",
    "log_fanout",
    "quiet_frac",
]

print("Configuration ready.")


Configuration ready.


## 1. Load the frozen scorer and test partition

Verify the upstream feature, representation and model-comparison state, then load the already-fitted K-means scorer. No `.fit()` call is made in this notebook.


In [2]:
def file_sha256(path):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


for required in [
    FEATURE_PATH,
    FEATURE_META_PATH,
    ACCOUNT_SPLIT_PATH,
    REPRESENTATION_META_PATH,
    CAL_METRICS_PATH,
    SCORERS_PATH,
    MODEL_META_PATH,
]:
    if not required.exists():
        raise FileNotFoundError(f"Missing required frozen input: {required}")

feature_meta = json.loads(FEATURE_META_PATH.read_text())
representation_meta = json.loads(REPRESENTATION_META_PATH.read_text())
model_meta = json.loads(MODEL_META_PATH.read_text())
split_obj = json.loads(ACCOUNT_SPLIT_PATH.read_text())

actual_hash = file_sha256(FEATURE_PATH)

if feature_meta["output_hashes"]["feat_final"] != actual_hash:
    raise RuntimeError("Canonical feature-table hash has changed.")

if representation_meta["core5_generic"] != CORE5:
    raise RuntimeError("CORE5 mismatch with notebook 01b.")

if model_meta["primary_method"] != PRIMARY_METHOD:
    raise RuntimeError("Unexpected primary scorer in model-comparison metadata.")

if model_meta["test_evaluated"] is not False:
    raise RuntimeError("Model-comparison stage unexpectedly evaluated the test partition.")

with open(SCORERS_PATH, "rb") as f:
    scorer_bundle = pickle.load(f)

primary_scorer = scorer_bundle["scorers"][PRIMARY_METHOD]

test_a = set(map(str, split_obj["test"]))

con = duckdb.connect()
feat = con.execute(
    f"SELECT * FROM '{FEATURE_PATH.as_posix()}'"
).df()
con.close()

feat["u"] = feat["u"].astype(str)
test_df = feat.loc[feat["u"].isin(test_a)].reset_index(drop=True)

print(f"Test rows / accounts: {len(test_df):,} / {test_df['u'].nunique():,}")
print("No model was fitted or refitted.")


Test rows / accounts: 26,115 / 7,391
No model was fitted or refitted.


## 2. Test window-0 performance

Evaluate the frozen K-means score using the same automation-associated reference definition as in calibration.


In [3]:
def evaluate_scores(df_with_labels, scores, ks=KS):
    scores = np.asarray(scores, dtype=float)

    reference = df_with_labels["is_anchor"].to_numpy(dtype=bool)
    human = df_with_labels["kind"].eq("human").to_numpy()
    eval_mask = reference | human

    out = {
        "reference_prevalence": float(reference[eval_mask].mean()),
        "AUPRC": float(
            average_precision_score(
                reference[eval_mask],
                scores[eval_mask],
            )
        ),
        "ROC_AUC": float(
            roc_auc_score(
                reference[eval_mask],
                scores[eval_mask],
            )
        ),
    }

    for k in ks:
        out[f"P@{k}"] = se.precision_at_k(scores, reference, k)
        out[f"R@{k}"] = se.recall_at_k(scores, reference, k)

    return out


test_labeled = se.add_labels(test_df)
test_w0 = (
    test_labeled.loc[test_labeled["win"] == STATIC_EVAL_WIN]
    .reset_index(drop=True)
)

test_scores = primary_scorer.score(test_w0)

test_metrics = pd.DataFrame([{
    "method": PRIMARY_METHOD,
    **evaluate_scores(test_w0, test_scores),
}])

test_metrics.to_csv(TEST_METRICS_PATH, index=False)

print(test_metrics[
    ["method", "AUPRC", "ROC_AUC", "P@30", "P@50", "P@100"]
].round(3).to_string(index=False))


      method  AUPRC  ROC_AUC  P@30  P@50  P@100
kmeans_core5  0.849    0.739 0.967  0.96   0.96


## 3. Calibration-to-test comparison

Compare the frozen K-means result with its calibration-window-0 performance. This reproduces dissertation Table 4.


In [4]:
cal_metrics = pd.read_csv(CAL_METRICS_PATH)

cal_row = cal_metrics.loc[
    cal_metrics["method"] == PRIMARY_METHOD
].iloc[0]

test_row = test_metrics.iloc[0]

comparison = pd.DataFrame([{
    "Partition": "Calibration window 0",
    "AP": cal_row["AUPRC"],
    "AUROC": cal_row["ROC_AUC"],
    "P@30": cal_row["P@30"],
    "P@50": cal_row["P@50"],
    "P@100": cal_row["P@100"],
}])

comparison = pd.concat([
    comparison,
    pd.DataFrame([{
        "Partition": "Test window 0",
        "AP": test_row["AUPRC"],
        "AUROC": test_row["ROC_AUC"],
        "P@30": test_row["P@30"],
        "P@50": test_row["P@50"],
        "P@100": test_row["P@100"],
    }])
], ignore_index=True)

print(comparison.round(3).to_string(index=False))


           Partition    AP  AUROC  P@30  P@50  P@100
Calibration window 0 0.853  0.746 1.000  0.98   0.94
       Test window 0 0.849  0.739 0.967  0.96   0.96


## 4. Test cross-window stability

Compare the frozen test-account ranking in window 0 with windows 1–3 using accounts present in both windows.


In [5]:
tmp = test_df[["u", "win"]].copy()
tmp["score"] = primary_scorer.score(test_df)

s0 = (
    tmp.loc[tmp["win"] == 0, ["u", "score"]]
    .rename(columns={"score": "score_w0"})
)

stability_rows = []

for w in [1, 2, 3]:
    sw = (
        tmp.loc[tmp["win"] == w, ["u", "score"]]
        .rename(columns={"score": "score_w"})
    )

    merged = s0.merge(sw, on="u")

    stability_rows.append({
        "comparison": f"w0_vs_w{w}",
        "n_accounts": len(merged),
        "spearman": float(
            merged["score_w0"].corr(
                merged["score_w"],
                method="spearman",
            )
        ),
    })

test_stability = pd.DataFrame(stability_rows)
test_stability.to_csv(TEST_STABILITY_PATH, index=False)

median_test_rho = float(test_stability["spearman"].median())

print(test_stability.round(3).to_string(index=False))
print("\nMedian test stability:", round(median_test_rho, 3))


comparison  n_accounts  spearman
  w0_vs_w1        6085     0.861
  w0_vs_w2        6034     0.758
  w0_vs_w3        5948     0.752

Median test stability: 0.758


## 5. Save dissertation Table 4 and provenance


In [6]:
cal_stability = pd.read_csv(
    MODEL_DIR / "calibration_score_stability.csv"
)

cal_median_rho = float(
    cal_stability.loc[
        cal_stability["method"] == PRIMARY_METHOD,
        "spearman",
    ].median()
)

comparison["Median rho"] = [
    cal_median_rho,
    median_test_rho,
]

comparison.to_csv(CAL_VS_TEST_PATH, index=False)

final_test_meta = {
    "pipeline_stage": "02c_final_static_test_evaluation",
    "primary_method": PRIMARY_METHOD,
    "core5": CORE5,
    "models_refitted": False,
    "feature_selection_reopened": False,
    "model_selection_reopened": False,
    "test_used_for_tuning_in_clean_pipeline": False,
    "test_history_note": (
        "The clean pipeline did not use the test partition for model development, "
        "but the same split had been inspected during earlier exploratory work."
    ),
    "feature_sha256": actual_hash,
    "account_split_sha256": file_sha256(ACCOUNT_SPLIT_PATH),
    "model_comparison_meta_sha256": file_sha256(MODEL_META_PATH),
    "outputs": {
        "test_metrics": str(TEST_METRICS_PATH),
        "table_4": str(CAL_VS_TEST_PATH),
        "test_stability": str(TEST_STABILITY_PATH),
    },
}

META_PATH.write_text(json.dumps(final_test_meta, indent=2))

print(comparison.round(3).to_string(index=False))
print("\nFinal static test evaluation complete.")


           Partition    AP  AUROC  P@30  P@50  P@100  Median rho
Calibration window 0 0.853  0.746 1.000  0.98   0.94       0.765
       Test window 0 0.849  0.739 0.967  0.96   0.96       0.758

Final static test evaluation complete.
